In this notebook, i will try to use

phase 1:
- Emoji Processing
- Lowecasing
- Text Variation Cleaning

phase 2:
- Tokenizing
- Stemming
- Slang Substitution
- Stopword Removal

## `Import Libraries`

In [ ]:
# importing libraries
import pandas as pd
import re
import unicodedata
from unidecode import unidecode

## `Load Dataset`

Using `data-v1.ipynb` dataset which has already balanced before.

In [3]:
# read dataset
df = pd.read_csv("data/data_balanced.csv")
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 8154 entries, 0 to 8153
Data columns (total 2 columns):
 #   Column    Non-Null Count  Dtype
---  ------    --------------  -----
 0   komentar  8154 non-null   str  
 1   label     8154 non-null   int64
dtypes: int64(1), str(1)
memory usage: 127.5 KB


## `Preprocessing Text`

In [ ]:
# phase 1:
# Emoji Processing
# Lowecasing
# Text Variation Cleaning

def normalize_all_cases(text):
    if not isinstance(text, str): return ""
    
    # Visual Mapping
    visual_map = {
        'Δ': 'a', 'ά': 'a', 'ᗩ': 'a',
        'ᗯ': 'w', 'vv': 'w',
        'σ': 'o', '𝐎': 'o', '0': 'o',
        '𝐍': 'n', 'И': 'n',
        '𝓽': 't', '𝓣': 't',
        '𝐒': 's', '🅢': 's',
    }
    
    for char, replacement in visual_map.items():
        text = text.replace(char, replacement)
    
    # Change symbol to ASCII
    text = unidecode(text)
    
    # Zalgo Normalization & Combining Marks (C̶ -> C)
    text = unicodedata.normalize('NFKD', text)
    text = "".join([c for c in text if unicodedata.category(c) != 'Mn'])
    
    # Lowercase & clean non-alfanumeric case
    text = text.lower()
    text = re.sub(r'[^a-z0-9\s]', ' ', text)
    
    # Aggressive Space Collapse
    while True:
        new_text = re.sub(r'(?<=\b[a-z])\s+(?=[a-z]\b)', '', text)
        if new_text == text: break
        text = new_text
    
    return " ".join(text.split()).strip()

# input_text = "🅿️🆄🅻🅰️🆄777"
# output = normalize_all_cases(input_text)

# print(output)

In [16]:
df_p1 = df.copy()
df_p1["komentar_cleaned_p1"] = df_p1["komentar"].apply(normalize_all_cases)
df_p1.to_csv("data/data_cleaned_p1.csv", index=False)
df_p1.info()

<class 'pandas.DataFrame'>
RangeIndex: 8154 entries, 0 to 8153
Data columns (total 3 columns):
 #   Column               Non-Null Count  Dtype
---  ------               --------------  -----
 0   komentar             8154 non-null   str  
 1   label                8154 non-null   int64
 2   komentar_cleaned_p1  8154 non-null   str  
dtypes: int64(1), str(2)
memory usage: 191.2 KB


In [ ]:
# phase 2:
# Tokenizing
# Slang Substitution
# Stemming
# Stopword Removal

import nltk
from nltk.tokenize import word_tokenize
nltk.download('punkt')
nltk.download('punkt_tab')
nltk.download('stopwords')

# load data
df_p2 = df_p1.copy()

# get text
texts = df_p2['komentar_cleaned_p1']
# convert text to list
texts = texts.tolist()
# apply tokenization for every sentence in text list
tokenized_text = [word_tokenize(sentence) for sentence in texts]
df_p2['tokenized_text_p2'] = tokenized_text

[nltk_data] Downloading package punkt to
[nltk_data]     /Users/walkervalentinuss/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package punkt_tab to
[nltk_data]     /Users/walkervalentinuss/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.
[nltk_data] Downloading package stopwords to
[nltk_data]     /Users/walkervalentinuss/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


In [19]:
df_p2.tail()

,komentar,label,komentar_cleaned_p1,tokenized_text_p2
8149,"Lebih baik nonton Pada Zaman dahulu 😭,",0,lebih baik nonton pada zaman dahulu,"[lebih, baik, nonton, pada, zaman, dahulu]"
8150,😂😂 udah kalok gak faham diem aja...,0,udah kalok gak faham diem aja,"[udah, kalok, gak, faham, diem, aja]"
8151,Apalagi ini 𝐁𝐔𝐊𝐈𝐓𝟖𝟖𝟖,1,apalagi ini bukit888,"[apalagi, ini, bukit888]"
8152,"Dah mo tutup dilernya\nJual kemahalan, pa ga c...",0,dah mo tutup dilernya jual kemahalan pa ga cpt...,"[dah, mo, tutup, dilernya, jual, kemahalan, pa..."
8153,Inilah pentingnya harus membiasakan baca buku...,0,inilah pentingnya harus membiasakan baca buku ...,"[inilah, pentingnya, harus, membiasakan, baca,..."


In [21]:
from Sastrawi.Stemmer.StemmerFactory import StemmerFactory
from nltk.corpus import stopwords

# Load Slang Dataset
slang_url = 'https://raw.githubusercontent.com/okkyibrohim/id-abusive-language-detection/master/kamusalay.csv'
slang_df = pd.read_csv(slang_url, encoding='latin-1', header=None, names=['slang', 'formal'])

# Convert to dictionary
slang_dict = dict(zip(slang_df['slang'], slang_df['formal']))

# Initialize Sastrawi
stemmer = StemmerFactory().create_stemmer()
stopword = set(stopwords.words('indonesian')) \
            | set(stopwords.words('english'))

In [24]:
# Input: token list from 'tokenized_text_p2'
df_p2['text_slang_p2'] = df_p2['tokenized_text_p2'].apply(
    lambda tokens: [slang_dict.get(w, w) for w in tokens]
)
# recombine for stopword stemming
df_p2['text_slang_str_p2'] = df_p2['text_slang_p2'].apply(lambda x: " ".join(x))

# stopword removal
def remove_stopwords(text):
    if not isinstance(text, str): return ""
    return " ".join([word for word in text.split() if word not in stopword])

df_p2['text_stopword_p2'] = df_p2['text_slang_str_p2'].apply(remove_stopwords)

# stemming, takes a lot of time
df_p2['text_stemmed_p2'] = df_p2['text_stopword_p2'].apply(lambda x: stemmer.stem(x))

# save dataset
df_p2.to_csv("data/data_cleaned_p2.csv", index=False)